# Langraph Tutorials

### 1. Installing Packages

In [ ]:
# Install required packages
!pip install langgraph langchain langchain-openai -q

### 2. Set OpenAI API Key

In [ ]:
from google.colab import userdata
import os

# === SET YOUR API KEY HERE ===
# Option 1: Using Colab Secrets (Recommended)
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Option 2: Manual input (uncomment if needed)
# os.environ["OPENAI_API_KEY"] = input("Enter your OpenAI API Key: ")

### 3. Imports & LLM Setup

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 4. Define Nodes

In [ ]:
def llm_node(state):
    """LLM Node - Generates response"""
    messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": messages + [response]}

def validator_node(state):
    """Simple validator node"""
    messages = state["messages"]
    last_content = messages[-1].content if messages else ""
    
    if "error" in last_content.lower():
        return {"error": "Validation failed", "next": "llm"}
    return {"error": None, "next": "end"}

### 5. Simple Linear Graph

In [ ]:
workflow = StateGraph()

workflow.add_node("llm", llm_node)
workflow.add_node("validator", validator_node)

workflow.add_edge(START, "llm")
workflow.add_edge("llm", "validator")
workflow.add_edge("validator", END)

basic_app = workflow.compile()

# Test
initial_state = {
    "messages": [HumanMessage(content="Hello! Tell me about yourself.")],
    "error": None,
    "next": ""
}

result = basic_app.invoke(initial_state)
print("Final Response:")
print(result["messages"][-1].content)

### 6. Graph with Loop (Self-Correction)

In [ ]:
def route_after_validator(state):
    if state.get("error"):
        return "llm"
    return END

workflow_cycle = StateGraph()

workflow_cycle.add_node("llm", llm_node)
workflow_cycle.add_node("validator", validator_node)

workflow_cycle.add_edge(START, "llm")
workflow_cycle.add_edge("llm", "validator")
workflow_cycle.add_conditional_edges(
    "validator",
    route_after_validator,
    {"llm": "llm", END: END}
)

cycle_app = workflow_cycle.compile()

# Test
result = cycle_app.invoke({
    "messages": [HumanMessage(content="Generate a structured list.")],
    "error": None,
    "next": ""
})

### 7. ReAct Agent

In [ ]:
@tool
def search_tool(query: str):
    """Dummy search tool"""
    return f"Search results for '{query}': Found relevant information about {query}."

tools = [search_tool]
llm_with_tools = llm.bind_tools(tools)

def agent_node(state):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": messages + [response]}

def tool_node(state):
    messages = state["messages"]
    last_message = messages[-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        results = []
        for tc in last_message.tool_calls:
            if tc["name"] == "search_tool":
                result = search_tool.invoke(tc["args"]["query"])
                results.append(result)
        return {"messages": messages + [HumanMessage(content=str(results))]}
    return {"messages": messages}

def should_continue(state):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# Build ReAct Graph
react_graph = StateGraph()

react_graph.add_node("agent", agent_node)
react_graph.add_node("tools", tool_node)

react_graph.add_edge(START, "agent")
react_graph.add_conditional_edges(
    "agent", 
    should_continue, 
    {"tools": "tools", END: END}
)
react_graph.add_edge("tools", "agent")  # Loop back

react_app = react_graph.compile()

### 8. Test the ReAct Agent

In [ ]:
# Test the agent
final_result = react_app.invoke({
    "messages": [HumanMessage(content="Search for what is LangGraph")]
})

print("Final Answer:")
print(final_result["messages"][-1].content)

### 9. Stream Output

In [ ]:
print("Streaming output:\n")
for chunk in react_app.stream(
    {"messages": [HumanMessage(content="What can LangGraph do?")]},
    stream_mode="values"
):
    if chunk.get("messages"):
        print(chunk["messages"][-1].content)
        print("-" * 60)